# Chapter 4: The Operational Layer

This notebook sets up the Lakebase (managed Postgres) operational layer for the vehicle tracking system. We create three tables -- `vehicles`, `vehicle_positions` and `trips` -- that hold the live transactional data written by the simulator in Chapter 5 and read by the Streamlit app in Chapter 6.

Connection details are obtained from the Lakebase Connect dialog in the Databricks UI. The OAuth token expires after one hour. For a long-running session, generate a fresh token from the Connect dialog and update `LAKEBASE_TOKEN` before reconnecting.

## 1. Install Dependencies

In [1]:
%pip install psycopg2-binary==2.9.12 \
             pyyaml==6.0.3 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import os
import psycopg2

from config_validator import load_config, ConfigError

## 3. Configuration

In [3]:
LAKEBASE_HOST   = os.environ["LAKEBASE_HOST"]
LAKEBASE_USER   = os.environ["LAKEBASE_USER"]
LAKEBASE_TOKEN  = os.environ["LAKEBASE_TOKEN"]
LAKEBASE_DBNAME = os.environ["LAKEBASE_DBNAME"]

try:
    cfg = load_config("config.yaml")
except (FileNotFoundError, ConfigError) as e:
    raise SystemExit(f"Config error: {e}")

print(f"City: {cfg['city']['name']}")
print("Credentials set.")

City: London Borough of Merton
Credentials set.


## 4. Connect to Lakebase

In [4]:
try:
    conn = psycopg2.connect(
        host     = LAKEBASE_HOST,
        user     = LAKEBASE_USER,
        password = LAKEBASE_TOKEN,
        dbname   = LAKEBASE_DBNAME,
        sslmode  = "require",
        port     = 5432
    )
    conn.autocommit = True
    cursor = conn.cursor()
    print("Connected to Lakebase.")
except psycopg2.OperationalError as e:
    if "OAuth" in str(e) or "not authorized" in str(e).lower():
        print("ERROR: Lakebase connection failed.")
        print("The OAuth token has expired or is invalid.")
        print("Generate a fresh token from the Lakebase Connect dialog")
        print("in the Databricks UI and update LAKEBASE_TOKEN.")
    else:
        raise

Connected to Lakebase.


## 5. Drop Existing Tables

In [5]:
cursor.execute("DROP TABLE IF EXISTS trips CASCADE")
cursor.execute("DROP TABLE IF EXISTS vehicle_positions CASCADE")
cursor.execute("DROP TABLE IF EXISTS vehicles CASCADE")

print("Existing tables dropped.")

Existing tables dropped.


## 6. Create Tables

In [6]:
# vehicles -- one row per vehicle, slow-changing
cursor.execute("""
    CREATE TABLE vehicles (
        vehicle_id  TEXT PRIMARY KEY,
        driver_name TEXT NOT NULL,
        zone        TEXT NOT NULL,
        status      TEXT NOT NULL DEFAULT 'idle',
        created_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )
""")

# vehicle_positions -- high-frequency inserts, one row per position update
cursor.execute("""
    CREATE TABLE vehicle_positions (
        position_id  BIGSERIAL PRIMARY KEY,
        vehicle_id   TEXT NOT NULL REFERENCES vehicles(vehicle_id),
        lat          DOUBLE PRECISION NOT NULL,
        lon          DOUBLE PRECISION NOT NULL,
        speed_kmh    DOUBLE PRECISION,
        current_zone TEXT,
        recorded_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
    )
""")

# trips -- one row per trip, tracks lifecycle from request to completion
cursor.execute("""
    CREATE TABLE trips (
        trip_id      TEXT PRIMARY KEY,
        vehicle_id   TEXT REFERENCES vehicles(vehicle_id),
        pickup_lat   DOUBLE PRECISION NOT NULL,
        pickup_lon   DOUBLE PRECISION NOT NULL,
        dropoff_lat  DOUBLE PRECISION NOT NULL,
        dropoff_lon  DOUBLE PRECISION NOT NULL,
        pickup_zone  TEXT,
        dropoff_zone TEXT,
        status       TEXT NOT NULL DEFAULT 'requested',
        requested_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
        started_at   TIMESTAMPTZ,
        completed_at TIMESTAMPTZ
    )
""")

print("Tables created.")

Tables created.


## 7. Create Indexes

In [7]:
# Fast lookup of latest positions per vehicle
cursor.execute("""
    CREATE INDEX idx_vehicle_positions_vehicle_id
    ON vehicle_positions (vehicle_id, recorded_at DESC)
""")

# Fast lookup of vehicles by status and zone (used by nearest-driver query)
cursor.execute("""
    CREATE INDEX idx_vehicles_status_zone
    ON vehicles (status, zone)
""")

# Fast lookup of trips by status
cursor.execute("""
    CREATE INDEX idx_trips_status
    ON trips (status)
""")

print("Indexes created.")

Indexes created.


## 8. Seed Vehicles

In [8]:
# Vehicles spread across the zones
vehicles = [
    (v["id"], v["driver"], v["zone"])
    for v in cfg["vehicles"]
]

cursor.executemany("""
    INSERT INTO vehicles (vehicle_id, driver_name, zone)
    VALUES (%s, %s, %s)
""", vehicles)

print(f"Seeded {len(vehicles)} vehicles.")

Seeded 10 vehicles.


## 9. Verify

In [9]:
# Table row counts
for table in ["vehicles", "vehicle_positions", "trips"]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    count = cursor.fetchone()[0]
    print(f"{table:<20} {count:,} rows")

print()

# Vehicles by zone
cursor.execute("""
    SELECT zone, COUNT(*) AS vehicles
    FROM vehicles
    GROUP BY zone
    ORDER BY zone
""")
print("Vehicles per zone:")
for row in cursor.fetchall():
    print(f"  {row[0]:<15} {row[1]}")

print()

# Sample vehicle records
cursor.execute("SELECT vehicle_id, driver_name, zone, status FROM vehicles ORDER BY vehicle_id LIMIT 5")
print("Sample vehicles:")
print(f"  {'ID':<8} {'Driver':<18} {'Zone':<15} {'Status'}")
for row in cursor.fetchall():
    print(f"  {row[0]:<8} {row[1]:<18} {row[2]:<15} {row[3]}")

vehicles             10 rows
vehicle_positions    0 rows
trips                0 rows

Vehicles per zone:
  Colliers Wood   2
  Mitcham         2
  Morden          2
  Raynes Park     2
  Wimbledon       2

Sample vehicles:
  ID       Driver             Zone            Status
  V001     Alice Mensah       Wimbledon       idle
  V002     Ben Okafor         Wimbledon       idle
  V003     Clara Singh        Raynes Park     idle
  V004     David Petrov       Raynes Park     idle
  V005     Elena Costa        Colliers Wood   idle


## 10. Teardown

In [10]:
cursor.close()
conn.close()
print("Connection closed.")

Connection closed.
